Cell 1: Environment and Drive Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import yfinance as yf
from google.colab import drive
from ipywidgets import interact, widgets

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Setup Standard Directory
base_dir = '/content/drive/MyDrive/MSFin_Python'
data_dir = os.path.join(base_dir, 'data', 'PS6_data_files')
os.makedirs(data_dir, exist_ok=True)

# --- OUTPUT CONFIG ---
OUTPUT_FILENAME = 'PS_combined_output.csv'
combined_output_path = os.path.join(data_dir, OUTPUT_FILENAME)

print(f"Environment Ready. Output will save to: {combined_output_path}")

Cell 2: Data Ingestion and Processing

In [ ]:
# --- DATA REPOSITORY - MAPPING FILE ---
MAPPING_FILE_ID = '1HK4EOve-qKPxDvGj8vD-Gjng0vT6Cf_j'

def load_mapping_data(file_id):
    url = f'https://drive.google.com/uc?export=download&id={file_id}'
    return pd.read_csv(url)

# --- DATA REPOSITORY - ETF TIME SERIES FILES ---
FILE_IDS = {
    'VIX': '1RyQ_5nB-aQCtHpV3ZUVUz4CxnmZhIBDZ', 'SPY': '1VmhgDXAHpf9_guZPH97Oqsuyc8M-XqWV', 'XLF': '1SNbxWRoPr_GMSEAn8ldPWcDWRb7lYV5g',
    'XLE': '1OCGwQxzVWch51Fe_fm5_3fttw63ZxWW5', 'XLK': '1LbJH8iwMMnbSkwL7tAV22fBXUmolvWLp', 'XLI': '1zsTSmhVe5AaSp-edBMQPe0Rhc0ZbjAT7',
    'XLU': '1S32exlq9NKreCiLO3ZIaLP5GPGIYPys3', 'XLY': '1CCJRjm6vfOX1Pdu9Zk5M_7AkvAqCD6B0', 'XLB': '1RPzR_DMeIemAHYmvaVBV2LZXis6Avyay',
    'XLP': '1VKWg9bMDmC0ihp2BEraj5a2MsjmRtaqY', 'XLV': '1frQPDyMnoLWPZCt240Uerof8rYlxV8Yr'
}

def load_instructor_data(ticker):
    file_id = FILE_IDS.get(ticker)
    url = f'https://drive.google.com/uc?export=download&id={file_id}'
    return pd.read_csv(url, index_col='Date', parse_dates=True)

# Load Mapping and Create a Dictionary for Display
mapping_df = load_mapping_data(MAPPING_FILE_ID)

def get_quarterly_metrics(df, spy_df, vix_df, ticker):
    combined = df.join(spy_df, lsuffix='_etf', rsuffix='_spy').join(vix_df).dropna()
    combined.index = pd.to_datetime(combined.index)

    stats = []
    for period, group in combined.groupby(combined.index.to_period('Q')):
        if len(group) < 40: continue

        # 1. Regime (Mean VIX)
        avg_vix = group['Close'].mean()
        state = 'Low' if avg_vix < 15 else 'Normal' if avg_vix < 20 else 'Elevated' if avg_vix < 25 else 'High'

        # 2. Returns & Risk
        ret_etf, ret_spy = group['Close_etf'].pct_change().dropna(), group['Close_spy'].pct_change().dropna()
        beta = np.cov(ret_etf, ret_spy)[0, 1] / np.var(ret_spy)
        vol = ret_etf.std() * np.sqrt(252)
        max_dd = ((1 + ret_etf).cumprod() / (1 + ret_etf).cumprod().cummax() - 1).min()

        stats.append({
            'Ticker': ticker, 'Quarter': str(period), 'Vol_State': state,
            'Volatility': round(vol, 4), 'Max_Drawdown': round(max_dd, 4), 'Beta': round(beta, 4)
        })
    return stats

# --- EXECUTE PROCESSING ---
print("Fetching and processing data...")
tickers = [t for t in FILE_IDS.keys() if t not in ['VIX', 'SPY']]
vix_anchor = load_instructor_data('VIX')

# Clean VIX data after loading to handle extraneous columns and NaT dates
vix_anchor = vix_anchor.loc[:, ~vix_anchor.columns.str.contains('^Unnamed')]
if 'Close' in vix_anchor.columns:
    vix_anchor = vix_anchor[['Close']]
vix_anchor = vix_anchor[vix_anchor.index.notna()]

spy_anchor = load_instructor_data('SPY')

all_results = []
for t in tickers:
    etf_df = load_instructor_data(t)
    all_results.extend(get_quarterly_metrics(etf_df, spy_anchor, vix_anchor, t))

final_df = pd.DataFrame(all_results)
final_df.to_csv(combined_output_path, index=False)
print(f"Success! Master file saved. Counts:\n{final_df.groupby('Vol_State')['Quarter'].count()}")

Cell 3: Interactive Risk Profile

In [ ]:
def plot_risk_profile(ticker):
    df = pd.read_csv(combined_output_path)
    etf_data = df[df['Ticker'] == ticker]
    sector_name = sector_map.get(ticker, "")

    metrics = ['Volatility', 'Max_Drawdown', 'Beta']
    states = ['All', 'Low', 'Normal', 'Elevated', 'High']
    colors = {'All': '#424242', 'Low': '#A9CCE3', 'Normal': '#95A5A6', 'Elevated': '#EB984E', 'High': '#C0392B'}

    fig, axes = plt.subplots(len(metrics), 1, figsize=(16, 14))
    fig.suptitle(f'Risk Environment Analysis: {ticker} - {sector_name}', fontsize=24, fontweight='bold', y=0.96)

    for i, metric in enumerate(metrics):
        ax = axes[i]
        for j, state in enumerate(states):
            subset = etf_data[metric] if state == 'All' else etf_data[etf_data['Vol_State'] == state][metric]
            if subset.empty: continue

            p10, p25, med, p75, p90 = subset.quantile([0.1, 0.25, 0.5, 0.75, 0.9])
            y_pos = len(states) - 1 - j

            # Plot "Slider"
            ax.hlines(y=y_pos, xmin=p10, xmax=p90, color=colors[state], linewidth=3, alpha=0.5)
            ax.hlines(y=y_pos, xmin=p25, xmax=p75, color=colors[state], linewidth=12, alpha=0.8)
            ax.plot(med, y_pos, '|', color='black', markersize=25, markeredgewidth=4)

            # Pinned Labels at Far Sides
            xmin, xmax = ax.get_xlim()
            ax.text(ax.get_xlim()[0], y_pos, f'Min = {subset.min():.2f}', va='center', ha='left',
                    fontsize=11, fontweight='bold', color='maroon', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
            ax.text(ax.get_xlim()[1], y_pos, f'Max = {subset.max():.2f}', va='center', ha='right',
                    fontsize=11, fontweight='bold', color='darkgreen', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

            # Median Label
            ax.text(med, y_pos + 0.35, f'{med:.2f}', ha='center', fontsize=12, fontweight='bold')

        ax.set_yticks(range(len(states)))
        ax.set_yticklabels(reversed(['ALL', 'Low', 'Normal', 'Elevated', 'High']), fontsize=13, fontweight='bold')
        ax.set_title(f'Metric: {metric}', loc='left', fontsize=16, fontweight='bold', pad=15)
        ax.tick_params(axis='x', labelsize=12)
        ax.grid(axis='x', linestyle='--', alpha=0.3)

    plt.tight_layout(rect=[0, 0.03, 1, 0.93])
    plt.show()

# Define sector_map from mapping_df for display
sector_map = mapping_df.set_index('Ticker')['Detail'].to_dict()

interact(plot_risk_profile, ticker=widgets.Dropdown(options=tickers, value='XLF'))